# start 

In [16]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import mean_squared_error, mean_absolute_error

# ============================================================
# SETTINGS
# ============================================================
SEASON = "winter"        # "summer" or "winter"

BASE_PATH = Path(
    r"C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\outputs"
)

MODELS = {
    "Chronos2": "chronos2",
    "TabICL": "tabicl",
    "Moirai2": "moirai2",
}

APPROACHES = ["AF", "FSWL", "RFFS", "SB", "Univariate"]
DAYS = [f"day{i}" for i in range(1, 6)]

PEAK_THRESHOLD = 2000

# ============================================================
# METRICS
# ============================================================
def RMSE(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def MAE(y_true, y_pred):
    return mean_absolute_error(y_true, y_pred)

def pRMSE(y_true, y_pred, threshold=2000):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    mask = y_true > threshold

    if mask.sum() == 0:
        return np.nan

    return np.sqrt(mean_squared_error(y_true[mask], y_pred[mask]))

def MBE(y_true, y_pred):
    return np.mean(y_pred - y_true)

# ============================================================
# EVALUATION
# ============================================================
results = []

for model_name, model_folder in MODELS.items():

    for approach in APPROACHES:

        rmse_scores = []
        mae_scores = []
        prmse_scores = []
        mbe_scores = []

        for day in DAYS:

            filename = f"{model_folder}_5min_{approach}_{SEASON}_{day}.csv"

            file_path = (
                BASE_PATH
                / model_folder
                / approach
                / filename
            )

            if not file_path.exists():
                print(f"Missing: {file_path}")
                continue

            df = pd.read_csv(file_path)

            y_true = df["actual"].values
            y_pred = df["prediction"].values

            rmse_scores.append(RMSE(y_true, y_pred))
            mae_scores.append(MAE(y_true, y_pred))
            prmse_scores.append(pRMSE(y_true, y_pred, PEAK_THRESHOLD))
            mbe_scores.append(MBE(y_true, y_pred))

        results.append({
            "Model": model_name,
            "Approach": approach,

            "RMSE": np.mean(rmse_scores),
            "RMSE_std": np.std(rmse_scores, ddof=1),

            "MAE": np.mean(mae_scores),
            "MAE_std": np.std(mae_scores, ddof=1),

            "pRMSE": np.mean(prmse_scores),
            "pRMSE_std": np.std(prmse_scores, ddof=1),

            "MBE": np.mean(mbe_scores),
            "MBE_std": np.std(mbe_scores, ddof=1),
        })

# ============================================================
# RESULTS
# ============================================================
results_df = pd.DataFrame(results)

metric_cols = results_df.columns.difference(["Model", "Approach"])
results_df[metric_cols] = results_df[metric_cols].round(2)

results_df = results_df.sort_values(
    ["Model", "Approach"]
).reset_index(drop=True)

print(results_df.to_string(index=False))

   Model   Approach    RMSE  RMSE_std     MAE  MAE_std   pRMSE  pRMSE_std     MBE  MBE_std
Chronos2         AF 1158.78    300.25  699.93   167.24 1149.76     298.17  169.30   370.40
Chronos2       FSWL 1156.44    300.61  697.75   163.51 1147.68     299.49  163.33   369.57
Chronos2       RFFS 1155.10    306.79  688.20   155.14 1146.38     307.24  131.87   388.47
Chronos2         SB 1142.41    337.69  707.92   192.13 1134.19     332.57  261.57   317.53
Chronos2 Univariate 1114.44    326.63  657.72   158.62 1106.26     329.38  126.16   303.65
 Moirai2         AF 1916.98    435.68 1159.64   327.04 1912.15     444.44 -555.86   466.54
 Moirai2       FSWL 1936.18    437.51 1160.03   335.43 1931.70     445.65 -623.18   442.45
 Moirai2       RFFS 2054.14    490.29 1210.54   386.00 2052.77     492.65 -935.20   446.86
 Moirai2         SB 1905.81    409.31 1150.93   266.12 1899.70     420.81 -520.04   512.03
 Moirai2 Univariate 1340.64    249.26  754.75   114.56 1334.52     255.21 -142.62   356.86

In [17]:
results_df

,Model,Approach,RMSE,RMSE_std,MAE,MAE_std,pRMSE,pRMSE_std,MBE,MBE_std
0,Chronos2,AF,1158.78,300.25,699.93,167.24,1149.76,298.17,169.30,370.40
1,Chronos2,FSWL,1156.44,300.61,697.75,163.51,1147.68,299.49,163.33,369.57
2,Chronos2,RFFS,1155.10,306.79,688.20,155.14,1146.38,307.24,131.87,388.47
3,Chronos2,SB,1142.41,337.69,707.92,192.13,1134.19,332.57,261.57,317.53
4,Chronos2,Univariate,1114.44,326.63,657.72,158.62,1106.26,329.38,126.16,303.65
5,Moirai2,AF,1916.98,435.68,1159.64,327.04,1912.15,444.44,-555.86,466.54
6,Moirai2,FSWL,1936.18,437.51,1160.03,335.43,1931.70,445.65,-623.18,442.45
7,Moirai2,RFFS,2054.14,490.29,1210.54,386.00,2052.77,492.65,-935.20,446.86
8,Moirai2,SB,1905.81,409.31,1150.93,266.12,1899.70,420.81,-520.04,512.03
9,Moirai2,Univariate,1340.64,249.26,754.75,114.56,1334.52,255.21,-142.62,356.86


In [18]:
# ============================================================
# EXPORT results_df TO LATEX TABLE WITH MODEL AS MULTIROW
# ============================================================

def make_latex_table(results_df, caption="Cross-validation forecasting results.", label="tab:cv_results"):

    df = results_df.copy()

    # Optional: enforce model order
    model_order = ["Chronos2", "Moirai2", "TabICL"]
    df["Model"] = pd.Categorical(df["Model"], categories=model_order, ordered=True)
    df = df.sort_values(["Model", "Approach"]).reset_index(drop=True)

    latex = []
    latex.append(r"\begin{table*}[!t]")
    latex.append(r"\centering")
    latex.append(r"\caption{" + caption + r"}")
    latex.append(r"\label{" + label + r"}")
    latex.append(r"\resizebox{\textwidth}{!}{%")
    latex.append(r"\begin{tabular}{llrrrrrrrr}")
    latex.append(r"\toprule")
    latex.append(
        r"Model & Approach & RMSE & RMSE$_{std}$ & MAE & MAE$_{std}$ & "
        r"pRMSE & pRMSE$_{std}$ & MBE & MBE$_{std}$ \\"
    )
    latex.append(r"\midrule")

    for model, group in df.groupby("Model", observed=True):
        n_rows = len(group)

        for i, (_, row) in enumerate(group.iterrows()):
            model_cell = rf"\multirow{{{n_rows}}}{{*}}{{{model}}}" if i == 0 else ""

            latex.append(
                f"{model_cell} & "
                f"{row['Approach']} & "
                f"{row['RMSE']:.2f} & "
                f"{row['RMSE_std']:.2f} & "
                f"{row['MAE']:.2f} & "
                f"{row['MAE_std']:.2f} & "
                f"{row['pRMSE']:.2f} & "
                f"{row['pRMSE_std']:.2f} & "
                f"{row['MBE']:.2f} & "
                f"{row['MBE_std']:.2f} \\\\"
            )

        latex.append(r"\midrule")

    latex[-1] = r"\bottomrule"
    latex.append(r"\end{tabular}%")
    latex.append(r"}")
    latex.append(r"\end{table*}")

    return "\n".join(latex)


latex_table = make_latex_table(
    results_df,
    caption="Cross-validation performance across models and feature-selection strategies for the summer test days.",
    label="tab:summer_cv_results"
)

print(latex_table)

\begin{table*}[!t]
\centering
\caption{Cross-validation performance across models and feature-selection strategies for the summer test days.}
\label{tab:summer_cv_results}
\resizebox{\textwidth}{!}{%
\begin{tabular}{llrrrrrrrr}
\toprule
Model & Approach & RMSE & RMSE$_{std}$ & MAE & MAE$_{std}$ & pRMSE & pRMSE$_{std}$ & MBE & MBE$_{std}$ \\
\midrule
\multirow{5}{*}{Chronos2} & AF & 1158.78 & 300.25 & 699.93 & 167.24 & 1149.76 & 298.17 & 169.30 & 370.40 \\
 & FSWL & 1156.44 & 300.61 & 697.75 & 163.51 & 1147.68 & 299.49 & 163.33 & 369.57 \\
 & RFFS & 1155.10 & 306.79 & 688.20 & 155.14 & 1146.38 & 307.24 & 131.87 & 388.47 \\
 & SB & 1142.41 & 337.69 & 707.92 & 192.13 & 1134.19 & 332.57 & 261.57 & 317.53 \\
 & Univariate & 1114.44 & 326.63 & 657.72 & 158.62 & 1106.26 & 329.38 & 126.16 & 303.65 \\
\midrule
\multirow{5}{*}{Moirai2} & AF & 1916.98 & 435.68 & 1159.64 & 327.04 & 1912.15 & 444.44 & -555.86 & 466.54 \\
 & FSWL & 1936.18 & 437.51 & 1160.03 & 335.43 & 1931.70 & 445.65 & -623.18 & 4

# end